In [0]:
#Top consecutive users

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Create a Spark session
spark = SparkSession.builder.appName("TopConsecutiveUsers").getOrCreate()

# Define the schema for your data
schema = ["user_id", "timestamp"]
data = [
    ("user1", "2023-08-26 10:10:00"),
    ("user1", "2023-08-26 10:10:25"),
    ("user2", "2023-08-26 12:00:00"),
    ("user2", "2023-08-26 12:10:00"),
    ("user1", "2023-08-27 14:30:00"),
    ("user1", "2023-08-28 16:00:00"),
    ("user2", "2023-08-27 16:30:00"),
    ("user1", "2023-08-29 18:00:00"),
]

# Create a DataFrame from the data
df = spark.createDataFrame(data, schema=schema)

# Convert the timestamp column to a date type
df = df.withColumn("date", F.to_date("timestamp"))

# Define a window specification to partition by user and order by date
window_spec = Window.partitionBy("user_id").orderBy("date")

# Calculate the difference between consecutive dates
df = df.withColumn("date_diff", F.datediff("date", F.lag("date").over(window_spec)))

# Identify consecutive login sequences
df = df.withColumn("is_consecutive", F.when(F.col("date_diff") == 1, 1).otherwise(0))

# Calculate the cumulative sum of consecutive logins
df = df.withColumn("consecutive_sum", F.sum("is_consecutive").over(window_spec))

# Find the maximum consecutive logins for each user
max_consecutive_df = df.groupBy("user_id").agg(F.max("consecutive_sum").alias("max_consecutive_days"))

# Find the top 5 users with the most consecutive logins
top_5_users = max_consecutive_df.orderBy(F.desc("max_consecutive_days")).limit(5)

# Show the result
top_5_users.show()


+-------+--------------------+
|user_id|max_consecutive_days|
+-------+--------------------+
|  user1|                   3|
|  user2|                   1|
+-------+--------------------+

